We create this code for the evaluation of rag applicatiosn using RAGAS(Retrieval Augmented Generation Assessment) open source framework..

In [3]:
# install packages 
!pip install "numpy>=1.26.4" -q
!pip install ragas==0.1.0 -q
!pip install langchain==0.3.2 -q
!pip install langchain-chroma -q
!pip install langchain-openai -q
!pip install datasets==2.16.1 -q
!pip install pypdf -q



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      + C:\Users\shara\AppData\Local\Programs\Python\Python313\python.exe C:\Users\shara\AppData\Local\Temp\pip-install-j_cpdafh\numpy_42f66f8de4cc45b09c11f127ebab579c\vendored-meson\meson\meson.py setup C:\Users\shara\AppData\Local\Temp\pip-install-j_cpdafh\numpy_42f66f8de4cc45b09c11f127ebab579c C:\Users\shara\AppData\Local\Temp\pip-install-j_cpdafh\numpy_42f66f8de4cc45b09c11f127ebab579c\.mesonpy-y0oxndrp -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\shara\AppData\Local\Temp\pip-install-j_cpdafh\numpy_42f66f8de4cc45b09c11f127ebab579c\.mesonpy-y0oxndrp\meson-py

In [2]:
#call the API key as an environment variable
#to manage API key as a local enviornment variable we need OS library, and to load the env variables from .env file we need to install python-dotenv package
%pip install python-dotenv

import os

#load openai key from .env file, first import the library to load env variables
from dotenv import load_dotenv
from pathlib import Path

env_path=Path.cwd() /'.env'#give the .env file path
print("Path to .env file:", env_path)
print("File exists:", env_path.exists())
# Load environment variables from .env file
load_dotenv(env_path,override=True)


#OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
api_key = os.getenv("OPENAI_API_KEY")
print("Loaded:", api_key is not None)
weather_key=os.getenv("OPEN_WEATHER_API_KEY")
print("Loaded:", weather_key is not None)


Path to .env file: c:\Users\shara\OneDrive\Documents\Coding Stuff\Generative AI\LangChain\.env
File exists: True
Loaded: True
Loaded: True



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
#initialize the chatOPENAI model
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.9, openai_api_key=api_key)

c:\Users\shara\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#intialize a embedding model
from langchain_openai import OpenAIEmbeddings 
embedding=OpenAIEmbeddings(model='text-embedding-3-small')

In [5]:
#Load pdf loader
from langchain_community.document_loaders import PyPDFLoader

#initialize the pdf document loader
loader=PyPDFLoader("/Users/shara/OneDrive/Documents/Coding Stuff/Generative AI/LangChain/Documents/global_warming.pdf")
pdf_data=loader.load()

print(pdf_data)

C:\Users\shara\AppData\Local\Temp\ipykernel_34544\935097732.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-11T11:26:44+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-06-11T11:26:44+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '/Users/shara/OneDrive/Documents/Coding Stuff/Generative AI/LangChain/Documents/global_warming.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content="Global Warming: An Overview\nWhat is Global Warming?\nGlobal warming refers to the long-term rise in Earth's average surface temperature, driven\nprimarily by the accumulation of greenhouse gases in the atmosphere. Since the\npre-industrial era around 1850, global average temperatures have increased by approximately\n1.1 degrees Celsius. The Intergovernmental Panel on Climate Change (IPCC) has confirmed\nthat this warming trend is unequivocally linked to human industrial activity, not natural climate\nvariability.\nGreenh

In [6]:
len(pdf_data)

3

In [7]:
#split the loaded pdf data into chunks using a splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

#intialize the text splitter with a chunck size of 300 and 50 overlap.
#here we use a chachter-level function to count the length of chunks(not the tiktoekn method at this moment)
text_splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=40,length_function=len)#which means we create chunks with chrhactersize of 50.

#splits
splits=text_splitter.split_documents(pdf_data)

In [8]:
len(splits)

22

In [9]:
#Create vector store 
from langchain_chroma import Chroma

#create a vector store from created docuemtn chunks
vectorstore=Chroma.from_documents(documents=splits,embedding=embedding)

In [10]:
#create the retirever
retriever=vectorstore.as_retriever(search_kwargs={"k":2})# retireve best 2 chunks

In [11]:
#define prompt template
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

#define a message tempalte for the chatbot
message="""Answer this question using provided context only. {question} Context: {context}"""

#Create a chat prompt template from the message
prompt=ChatPromptTemplate.from_messages([("human",message)])


In [12]:
from langchain_core.output_parsers import StrOutputParser

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [13]:
#invoke the chain with a example question
response=chain.invoke("what is global warming")
print(response)

Global warming refers to the long-term rise in Earth's average surface temperature, primarily driven by the accumulation of greenhouse gases in the atmosphere.


Evalaute the prepared RAG application

In [14]:
import pandas as pd
test_data = pd.read_csv(r"C:\Users\shara\OneDrive\Documents\Coding Stuff\Generative AI\LangChain\Documents\rag_evaluation_qa.csv")
test_data

,Question,Answer
0,By how much has the global average temperature...,Approximately 1.1 degrees Celsius since around...
1,What is the current atmospheric concentration ...,"Over 421 ppm, the highest in at least 3 millio..."
2,What was the pre-industrial level of atmospher...,Approximately 280 ppm.
3,What percentage of total greenhouse gas emissi...,About 76% of total global greenhouse gas emiss...
4,How many times more potent is methane than CO2...,Approximately 28 times more potent.
5,What is the largest source of global greenhous...,"Burning fossil fuels for energy and transport,..."
6,What percentage of emissions comes from defore...,About 10 to 12% of annual CO2 emissions.
7,How much has global sea level risen since 1900?,"Approximately 20 cm, currently rising at 3.6 m..."
8,By what percentage has Arctic sea ice Septembe...,Around 13% per decade.
9,How much more likely are extreme heat events c...,Five times more likely.


In [15]:
questions=test_data["Question"].to_list()
ground_truth=test_data["Answer"].to_list()

In [16]:
#prepare a dictionary for our test data details. we can alrdy fill it with ground truthe since we know it for each question.
data={"question":[],"answer":[],"contexts":[],"ground_truth":ground_truth}

In [17]:
data

{'question': [],
 'answer': [],
 'contexts': [],
 'ground_truth': ['Approximately 1.1 degrees Celsius since around 1850.',
  'Over 421 ppm, the highest in at least 3 million years.',
  'Approximately 280 ppm.',
  'About 76% of total global greenhouse gas emissions.',
  'Approximately 28 times more potent.',
  'Burning fossil fuels for energy and transport, contributing approximately 75% of global emissions.',
  'About 10 to 12% of annual CO2 emissions.',
  'Approximately 20 cm, currently rising at 3.6 mm per year.',
  'Around 13% per decade.',
  'Five times more likely.',
  'Thresholds beyond which large-scale changes become self-sustaining and irreversible.',
  'Approximately 1.5 trillion tonnes of frozen organic carbon.',
  'By around 2050, according to the IPCC.',
  'By 295 GW in 2022 alone.',
  'Around 10,000 tonnes of CO2 per year globally as of 2023.',
  'To limit warming to well below 2 degrees Celsius, with efforts toward 1.5 degrees.',
  'Approximately 2.5 to 2.9 degrees Celsi

In [18]:
#fill the dictionary with the qustion, and apppropriate answer from the RAG chain we created.  and also we add the context with the retrival chunks form the vector database for each question.
for query in questions:
    data["question"].append(query)
    data["answer"].append(chain.invoke(query))
    data["contexts"].append([doc.page_content for doc in retriever.invoke(query)])

In [19]:
data

{'question': ['By how much has the global average temperature risen since the pre-industrial era?',
  'What is the current atmospheric concentration of CO2 as of 2023?',
  'What was the pre-industrial level of atmospheric CO2?',
  'What percentage of total greenhouse gas emissions does CO2 account for?',
  'How many times more potent is methane than CO2 over a 100-year period?',
  'What is the largest source of global greenhouse gas emissions?',
  'What percentage of emissions comes from deforestation and land use change?',
  'How much has global sea level risen since 1900?',
  'By what percentage has Arctic sea ice September extent declined?',
  'How much more likely are extreme heat events compared to the pre-industrial era?',
  'What are climate tipping points?',
  'How much CO2 could be released from thawing permafrost regions?',
  'By when must global CO2 emissions reach net-zero to limit warming to 1.5 degrees Celsius?',
  'How much did global renewable electricity capacity grow 

In [21]:
#make dictionar in the dataset format. Because a dataset object is easy to
from datasets import Dataset
dataset=Dataset.from_dict(data)

In [22]:
#check the dataset
dataset[5]

{'question': 'What is the largest source of global greenhouse gas emissions?',
 'answer': 'The largest source of global greenhouse gas emissions is burning fossil fuels for energy and transport, contributing approximately 75% of emissions.',
 'contexts': ['CO2 over a 100-year period.\nMain Causes\nBurning fossil fuels for energy and transport is the largest source, contributing approximately\n75% of global greenhouse gas emissions. Deforestation adds another 10 to 12%, as trees',
  'that once absorbed CO2 are cleared, mainly for agriculture. The agriculture sector itself,\nthrough livestock farming and fertiliser use, contributes a further 10 to 12% of emissions.\nIndustrial processes such as cement and steel production account for roughly 5% of global\nemissions.'],
 'ground_truth': 'Burning fossil fuels for energy and transport, contributing approximately 75% of global emissions.'}

In [23]:
#Now we use the ragas to evlauaet the dataset
from ragas.metrics import context_precision,context_recall,answer_relevancy,faithfulness
from ragas import evaluate

result=evaluate(dataset=dataset,metrics=[context_precision,context_recall,answer_relevancy,faithfulness])

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [ ]:
result

In [ ]:
results=result.to_pandas()

In [24]:
from ragas.metrics import context_precision, context_recall, answer_relevancy, faithfulness
from ragas import evaluate

print("Ragas imported successfully")

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'